# Titanic Dataset 기반 Feature Engineering 파이프라인 비교 분석

- 문제 유형: Classification
- 목표 변수: `Survived`
- 비교 항목: 결측치 처리, 인코딩, 스케일링, 파생 변수 생성, Feature Selection
- 사용 모델: Logistic Regression, Random Forest
- 가산점 요소: `Pipeline`, `GridSearchCV`, Feature Importance 시각화


## STEP 01. 데이터 준비

Kaggle Titanic Dataset 형식의 `train.csv`를 사용한다. 로컬 데이터가 없으면 공개 mirror URL에서 다운로드하도록 구성하였다.


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebook':
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.append(str(PROJECT_DIR / 'src'))

from pipeline_experiment import *

df = load_data(PROJECT_DIR / 'data' / 'train.csv')
print('데이터 shape:', df.shape)
df.head()

In [ ]:
column_desc = pd.read_csv(PROJECT_DIR / 'results' / 'column_description.csv') if (PROJECT_DIR / 'results' / 'column_description.csv').exists() else None
if column_desc is None:
    make_basic_tables(df)
    column_desc = pd.read_csv(PROJECT_DIR / 'results' / 'column_description.csv')
column_desc

## STEP 02. 탐색적 데이터 분석(EDA)

필수 분석인 결측치 비율, 이상치, 변수 분포, 상관관계, 타겟 분포를 확인한다.


In [ ]:
# 결측치 비율
missing_ratio = df.isna().mean().sort_values(ascending=False) * 100
missing_ratio.to_frame('missing_ratio_percent')

In [ ]:
# 타겟 분포
survived_counts = df['Survived'].value_counts().sort_index()
survived_counts.to_frame('count')

In [ ]:
# EDA 시각화 생성
make_eda_figures(df)
print('EDA figures saved to:', PROJECT_DIR / 'figures')

In [ ]:
from IPython.display import Image, display
for name in ['eda_missing_ratio.png', 'eda_target_countplot.png', 'eda_age_histogram.png', 'eda_fare_boxplot.png', 'eda_correlation_heatmap.png']:
    display(Image(filename=str(PROJECT_DIR / 'figures' / name)))

## STEP 03. 특성 공학 파이프라인 구현

생존 예측에 도움이 될 수 있는 파생 변수를 생성한다.

생성 변수:
- `FamilySize`: 함께 탑승한 가족 수
- `IsAlone`: 혼자 탑승 여부
- `FarePerPerson`: 1인당 운임
- `Title`: 이름에서 추출한 호칭
- `CabinKnown`, `Deck`: 객실 정보 존재 여부 및 Deck
- `AgeGroup`: 나이 구간


In [ ]:
df_fe = add_derived_features(df)
df_fe[['Name', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Title', 'CabinKnown', 'Deck', 'AgeGroup']].head()

### 실험 조합

| 실험 | 결측치 처리 | 인코딩 | 스케일링 | Feature Selection |
|---|---|---|---|---|
| Base | 없음 | 없음 | 없음 | 없음 |
| Exp-1 | Mean | One-Hot | Standard | X |
| Exp-2 | Median | Label/Ordinal | MinMax | O |
| Exp-3 | Most Frequent | One-Hot | Robust | O |


## STEP 04-05. Feature Selection, 모델 학습 및 평가

`Pipeline` 객체를 사용해 전처리, 변수 선택, 모델 학습을 하나의 흐름으로 묶었다. 평가 지표는 Accuracy, Precision, Recall, F1-score, ROC-AUC를 사용한다.


In [ ]:
results, fitted_pipelines, grid_result, importance_df = run_experiments(df)
results[['Experiment', 'Model', 'Missing', 'Encoding', 'Scaling', 'FeatureSelection', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']]

In [ ]:
# Feature Selection 적용 전/후 비교: Exp-1과 Exp-3 비교
fs_compare = pd.read_csv(PROJECT_DIR / 'results' / 'feature_selection_compare.csv')
fs_compare[['Experiment', 'Model', 'FeatureSelection', 'Accuracy', 'F1', 'ROC_AUC']]

In [ ]:
# Random Forest Feature Importance 상위 변수
importance_df.head(15)

In [ ]:
display(Image(filename=str(PROJECT_DIR / 'figures' / 'feature_importance_top15.png')))

## GridSearchCV 추가 실험

가산점 항목을 위해 Exp-3 + RandomForest 조합에 대해 GridSearchCV를 적용하였다.


In [ ]:
grid_result

## 최종 해석

- Base 모델은 수치형 기본 변수만 사용했기 때문에 성능이 낮게 나타났다.
- Exp-1은 Mean Imputation, One-Hot Encoding, StandardScaler, 파생 변수를 함께 사용하여 Logistic Regression에서 가장 높은 ROC-AUC를 보였다.
- Label/Ordinal Encoding은 범주형 변수에 순서 관계를 부여할 수 있어 One-Hot 대비 성능이 제한될 수 있다.
- Feature Selection은 변수를 줄여 모델을 단순화하는 장점이 있으나, 본 실험에서는 전체 변수를 사용하는 Exp-1 Logistic Regression이 가장 안정적이었다.
- Random Forest Feature Importance 기준으로 Age, FarePerPerson, Fare, Title, Sex 관련 변수가 생존 예측에 중요한 변수로 나타났다.
